## Patch features
Computes patch-level metrics for a single chip: per-patch valid fraction, raw HH/HV backscatter statistics (`hh_mean`, `hv_mean`, `hh_std`, `hv_std`, `hv_hh_ratio`), ancillary centroid sampling (AMSR2, ERA5, distance-to-land), and `ia_mean` (Block 5 incidence angle).

Uses the already-built `training.data_loader` (B2.1) and `training.encoding` (B2.2) to get a real `Chip` and its embeddings.

In [4]:
from pathlib import Path

import numpy as np

from training import (
    ALL_BANDS,
    AMSR2_BANDS,
    ERA5_BANDS,
    GRID_SIZE,
    PATCH_SIZE,
    encode_chip,
    load_band_means,
    load_clay_module,
    load_metadata,
    load_scene,
    sar_metadata_from_entry,
    select_device,
    yield_chips,
)

### Configuration

In [5]:
BUCKET = "prescient-ice-data"
STATS_KEY = "training_data/ai4arctic/statistics/dataset_stats.json"
AWS_PROFILE = "spk_data"

SCENE_PATH = Path(
    "../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75"
    "_icechart_dmi_201801241950_SouthEast_RIC.nc"
)
METADATA_PATH = Path("../../configs/metadata.yaml")
CHECKPOINT_PATH = Path("../../clay-v1.5.ckpt")

### Load one scene and get a single chip
`load_scene`/`yield_chips` are the B2.1 data loader — same as `chip_patch_prep.ipynb`.

In [6]:
band_means = load_band_means(BUCKET, STATS_KEY, ALL_BANDS, profile=AWS_PROFILE)
scene = load_scene(SCENE_PATH, band_means)
chip = next(yield_chips(scene))

print(f"chip_id: {chip.chip_id}")
print(f"valid fraction: {chip.valid_mask.mean():.3f}")

/Users/yolandajian/arctic-showcase/src/training/src/training/data_loader/labels.py:32: RuntimeWarning: invalid value encountered in cast
  poly_ids_int = np.where(valid_polygon, poly_chart.astype(np.int32), 0)


chip_id: S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC_r00000_c00000
valid fraction: 0.818


### Patch-features function
Subdivides the chip into the 32×32 patch grid. `chip.sar` is `(2, 256, 256)` with HH at index 0 and HV at index 1; `chip.amsr2`/`chip.era5` are stacked in `AMSR2_BANDS`/`ERA5_BANDS` order (see `training.data_loader.bands`). Valid fraction and the SAR statistics use only pixels flagged valid by B2.1's mask; the ancillary centroid samples and `ia_mean` don't filter on validity beyond `ia_mean`'s own patch-level mask.

In [7]:
def compute_patch_features(chip):
    """
    Given a Chip from yield_chips(), compute patch-level features for all
    1024 patches in the 32x32 grid.

    Returns a list of 1024 dicts, one per patch, in row-major order.
    """
    valid = chip.valid_mask  # (256, 256) bool
    hh = chip.sar[0]  # (256, 256)
    hv = chip.sar[1]  # (256, 256)
    distance = chip.distance_map  # (256, 256)
    ia = chip.incidence_angle  # (256, 256)

    patch_records = []

    for pi in range(GRID_SIZE):
        for pj in range(GRID_SIZE):
            # Pixel slice for this patch
            r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
            c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE

            valid_patch = valid[r0:r1, c0:c1]  # (8, 8) bool
            hh_patch = hh[r0:r1, c0:c1]  # (8, 8)
            hv_patch = hv[r0:r1, c0:c1]  # (8, 8)

            # Valid fraction: count over the boolean mask (64 pixels per 8x8 patch)
            valid_frac = valid_patch.sum() / 64.0

            # SAR statistics over valid pixels only
            hh_valid = hh_patch[valid_patch]
            hv_valid = hv_patch[valid_patch]

            if len(hh_valid) > 0:
                hh_mean = float(hh_valid.mean())
                hh_std = float(hh_valid.std())
                hv_mean = float(hv_valid.mean())
                hv_std = float(hv_valid.std())
                # Values are in dB, so subtract rather than divide for the ratio
                hv_hh_ratio = float((hv_valid - hh_valid).mean())
            else:
                hh_mean = hh_std = hv_mean = hv_std = hv_hh_ratio = np.nan

            # Patch centroid pixel coordinates
            rc = r0 + PATCH_SIZE // 2
            cc = c0 + PATCH_SIZE // 2

            # Ancillary centroid samples -- AMSR2/ERA5 already resampled to SAR
            # resolution by B2.1, so this is a direct sample, no averaging
            amsr2_sample = {var: float(chip.amsr2[i, rc, cc]) for i, var in enumerate(AMSR2_BANDS)}
            era5_sample = {var: float(chip.era5[i, rc, cc]) for i, var in enumerate(ERA5_BANDS)}

            dist = float(distance[rc, cc])
            ia_mean = (
                float(ia[r0:r1, c0:c1][valid_patch].mean()) if valid_patch.any() else np.nan
            )

            record = {
                "chip_id": chip.chip_id,
                "patch_i": pi,
                "patch_j": pj,
                "valid_frac": valid_frac,
                "hh_mean": hh_mean,
                "hv_mean": hv_mean,
                "hh_std": hh_std,
                "hv_std": hv_std,
                "hv_hh_ratio": hv_hh_ratio,
                "ia_mean": ia_mean,
                "distance": dist,
                **amsr2_sample,
                **era5_sample,
            }

            patch_records.append(record)

    return patch_records

### Compute patch features for the chip

In [8]:
patches = compute_patch_features(chip)
print(f"Patches: {len(patches)}")  # should be 1024 (32 x 32)
print(patches[0])

Patches: 1024
{'chip_id': 'S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC_r00000_c00000', 'patch_i': 0, 'patch_j': 0, 'valid_frac': np.float64(0.0), 'hh_mean': nan, 'hv_mean': nan, 'hh_std': nan, 'hv_std': nan, 'hv_hh_ratio': nan, 'ia_mean': nan, 'distance': 35.0, 'btemp_6_9h': 148.78390502929688, 'btemp_6_9v': 203.45779418945312, 'btemp_7_3h': 0.0, 'btemp_7_3v': 0.0, 'btemp_10_7h': 0.0, 'btemp_10_7v': 0.0, 'btemp_18_7h': 165.60284423828125, 'btemp_18_7v': 218.53102111816406, 'btemp_23_8h': 0.0, 'btemp_23_8v': 0.0, 'btemp_36_5h': 185.1192169189453, 'btemp_36_5v': 228.15660095214844, 'btemp_89_0h': 212.46035766601562, 'btemp_89_0v': 241.20924377441406, 'u10m_rotated': 0.6822869777679443, 'v10m_rotated': 0.6134480237960815, 't2m': 268.6146545410156, 'skt': 268.9005432128906, 'tcwv': 7.7288408279418945, 'tclw': 0.04101099818944931}


### Spot-check patch [2, 3] against a hand-counted valid fraction
Manually recomputes the valid fraction for one patch directly from `chip.valid_mask` and compares it against what `compute_patch_features` produced.

In [9]:
pi, pj = 2, 3
r0, r1 = pi * PATCH_SIZE, (pi + 1) * PATCH_SIZE
c0, c1 = pj * PATCH_SIZE, (pj + 1) * PATCH_SIZE

manual_valid_frac = chip.valid_mask[r0:r1, c0:c1].sum() / 64.0
module_valid_frac = patches[pi * GRID_SIZE + pj]["valid_frac"]

print(f"Manual valid_frac: {manual_valid_frac:.4f}")
print(f"Module valid_frac: {module_valid_frac:.4f}")
print(f"Match: {np.isclose(manual_valid_frac, module_valid_frac)}")

Manual valid_frac: 0.0000
Module valid_frac: 0.0000
Match: True


### Look up and visualize a patch
Change `PATCH_I`/`PATCH_J` (each in `[0, 32)`) to inspect a different patch. Shows the record, the full chip's HH band with the patch outlined, and a zoomed-in crop of just that patch.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt

PATCH_I, PATCH_J = 0, 7  # change these to inspect a different patch

patch_idx = PATCH_I * GRID_SIZE + PATCH_J
record = patches[patch_idx]
print({k: v for k, v in record.items() if k != "embedding"})

r0, r1 = PATCH_I * PATCH_SIZE, (PATCH_I + 1) * PATCH_SIZE
c0, c1 = PATCH_J * PATCH_SIZE, (PATCH_J + 1) * PATCH_SIZE

vmin, vmax = np.percentile(chip.sar[0], [2, 98])

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(chip.sar[0], cmap="gray", vmin=vmin, vmax=vmax)
axes[0].add_patch(
    mpatches.Rectangle(
        (c0, r0), PATCH_SIZE, PATCH_SIZE, edgecolor="red", facecolor="none", linewidth=1.5
    )
)
axes[0].set_title(f"{chip.chip_id}\nHH, patch [{PATCH_I}, {PATCH_J}] outlined")

axes[1].imshow(chip.sar[0][r0:r1, c0:c1], cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title(f"Patch [{PATCH_I}, {PATCH_J}]  (valid_frac={record['valid_frac']:.2f})")

plt.tight_layout()
plt.show()

### Attach Clay embeddings to each patch (B2.2 + B2.3 integration)
Loads Clay and encodes the same chip. `encode_chip` returns `patch_tokens` as `(1, 1024, 32, 32)` — channel-first, batch then embedding-dim then the 32×32 grid — so indexing a patch's 1024-dim vector is `patch_tokens[0, :, pi, pj]`, **not** `patch_tokens[pi, pj, :]`.

In [10]:
metadata = load_metadata(METADATA_PATH)
sar_meta = sar_metadata_from_entry(metadata["sentinel-1-ew"])
device = select_device()
module = load_clay_module(CHECKPOINT_PATH, METADATA_PATH, device)

patch_tokens, class_token = encode_chip(chip, module, sar_meta, device)

for pi in range(GRID_SIZE):
    for pj in range(GRID_SIZE):
        idx = pi * GRID_SIZE + pj
        patches[idx]["embedding"] = patch_tokens[0, :, pi, pj]

chip_row = {
    "chip_id": chip.chip_id,
    "lat": chip.centroid_lat,
    "lon": chip.centroid_lon,
    "embedding": class_token[0],
}

print(f"Chip row: {chip_row['chip_id']}")
print(f"Chip embedding shape:  {chip_row['embedding'].shape}")
print(f"Patches: {len(patches)}")
print(f"Patch embedding shape: {patches[0]['embedding'].shape}")

Chip row: S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75_icechart_dmi_201801241950_SouthEast_RIC_r00000_c00000
Chip embedding shape:  (1024,)
Patches: 1024
Patch embedding shape: (1024,)
